In [ ]:
# https://www.kaggle.com/competitions/orbit-wars/overview

In [19]:
!pip install --upgrade "kaggle-environments>=1.28.0" -q

In [20]:
import math, random, json, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical
from collections import deque
from kaggle_environments import make
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [21]:
ID, OWNER, X, Y, RADIUS, SHIPS, PROD = 0, 1, 2, 3, 4, 5, 6
SUN_X, SUN_Y, SUN_R = 50.0, 50.0, 10.0

In [22]:
env = make("orbit_wars", debug=False)

In [23]:
class Environment:
    def __init__(self):
        self.env = make("orbit_wars", debug=False)

    def compute_reward(self, prev_obs, new_obs):
        player = new_obs.get("player", 0)
        prev_planets = prev_obs.get("planets", [])
        new_planets = new_obs.get("planets", [])

        prev_owned = {p[ID] for p in prev_planets if p[OWNER] == player}
        new_owned = {p[ID] for p in new_planets if p[OWNER] == player}
        prev_enemy = {p[ID] for p in prev_planets if p[OWNER] != player and p[OWNER] != -1}
        new_enemy = {p[ID] for p in new_planets if p[OWNER] != player and p[OWNER] != -1}

        conquered = new_owned - prev_owned
        from_enemy = conquered & prev_enemy
        from_neutral = conquered - from_enemy

        conquest_enemy = sum(p[PROD] for p in new_planets if p[ID] in from_enemy)
        conquest_neutral = sum(p[PROD] for p in new_planets if p[ID] in from_neutral)
        prod = sum(p[PROD] for p in new_planets if p[OWNER] == player) - sum(p[PROD] for p in new_planets if p[OWNER] != player and p[OWNER] != -1)
        ships = sum(p[SHIPS] for p in new_planets if p[OWNER] == player) - sum(p[SHIPS] for p in new_planets if p[OWNER] != player and p[OWNER] != -1)

        return (
            np.log1p(conquest_enemy) * 0.75
            + np.log1p(conquest_neutral) * 0.15
            + np.log1p(max(0, prod)) * 0.05
            + np.log1p(max(0, ships)) * 0.05
        )

    def __call__(self, P1, P2):
        self.env.reset()
        prev_obs_1 = self.env.state[0].observation
        prev_obs_2 = self.env.state[1].observation
    
        while True:
            obs_1 = self.env.state[0].observation
            obs_2 = self.env.state[1].observation
            
            action_1 = P1(obs_1)
            action_2 = P2(obs_2)
            
            self.env.step([action_1, action_2])
            
            new_obs_1 = self.env.state[0].observation
            new_obs_2 = self.env.state[1].observation
            
            r1 = self.compute_reward(prev_obs_1, new_obs_1)
            r2 = self.compute_reward(prev_obs_2, new_obs_2)
            
            prev_obs_1 = new_obs_1
            prev_obs_2 = new_obs_2
            
            done = self.env.state[0].status != "ACTIVE"
            
            yield r1, r2, done
            
            if done:
                break

In [24]:
class ActorCritic(nn.Module):
    def __init__(self, input_size=512, hidden_size=256):
        super().__init__()

        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)

        self.actor_planet = nn.Linear(hidden_size, 60)
        self.actor_angle_mean = nn.Linear(hidden_size, 1)
        self.actor_angle_std = nn.Linear(hidden_size, 1)
        self.actor_ships_alpha = nn.Linear(hidden_size, 1)
        self.actor_ships_beta = nn.Linear(hidden_size, 1)
        self.critic = nn.Linear(hidden_size, 1)

    def forward(self, x, hidden=None):
        lstm_out, hidden = self.lstm(x, hidden)
        h = lstm_out[:, -1, :]
    
        planet_logit = self.actor_planet(h)
        angle_mean = self.actor_angle_mean(h)
        angle_std = F.softplus(self.actor_angle_std(h)) + 1e-4
        ships_alpha = F.softplus(self.actor_ships_alpha(h)) + 1e-4
        ships_beta = F.softplus(self.actor_ships_beta(h)) + 1e-4
        value = self.critic(h)
    
        return planet_logit, angle_mean, angle_std, ships_alpha, ships_beta, value, hidden

In [25]:
class A2CAgent:
    def __init__(self, n_games, input_size=512, hidden_size=256, lr=1e-3, gamma=0.99, entropy_coef=0.01, value_coef=0.5):
        self.model = ActorCritic(input_size, hidden_size).to(device)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=n_games, eta_min=1e-5)
        self.gamma   = gamma
        self.hidden  = None
        self.memory  = []
        self.entropy_coef = entropy_coef
        self.value_coef = value_coef

    def save(self, path):
        torch.save(self.model.state_dict(), path)

    def load(self, path):
        self.model.load_state_dict(torch.load(path))
        
    def __call__(self, obs):
        planets = obs.get("planets", [])
        player  = obs.get("player", 0)
        owned = [p[ID] for p in planets if p[OWNER] == player]
    
        if not owned:
            return []
            
        features = self.gen_features(obs)
        X = torch.FloatTensor(features).unsqueeze(0).unsqueeze(0).to(device)
        
        planet_logit, angle_mean, angle_std, ships_alpha, ships_beta, value, self.hidden = self.model(X, self.hidden)

        # Planet ID
        mask = torch.full((60,), float('-inf')).to(device)
        mask[owned] = 0.0
        logits = planet_logit.squeeze(0) + mask
        probs = torch.softmax(logits, dim=0)
        dist_planet = torch.distributions.Categorical(probs)
        idx = dist_planet.sample()
        src = planets[idx.item()]

        # Angles
        dist_angle = torch.distributions.Normal(angle_mean, angle_std)
        angle = dist_angle.sample().item()

        # Ships
        dist_ships = torch.distributions.Beta(ships_alpha, ships_beta)
        frac = dist_ships.sample().item()
        ships = max(1, int(frac * src[SHIPS]))
        
        log_prob = (
            dist_planet.log_prob(idx)
            + dist_angle.log_prob(torch.tensor(angle, device=device))
            + dist_ships.log_prob(torch.tensor(frac, device=device))
        )
        entropy  = dist_planet.entropy() + dist_angle.entropy() + dist_ships.entropy()

        self.memory.append((log_prob, value, None, entropy))
    
        return [[src[ID], angle, ships]]

    def store_reward(self, reward):
        if self.memory:
            lp, v, _, ent = self.memory[-1]
            self.memory[-1] = (lp, v, reward, ent)

    def update(self, next_value=0.0):
        if not self.memory:
            return

        R = next_value
        returns = []
        for _, _, r, _ in reversed(self.memory):
            R = (r or 0.0) + self.gamma * R
            returns.insert(0, R)

        returns = torch.FloatTensor(returns).to(device)
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        actor_loss  = 0.0
        critic_loss = 0.0
        entropy_loss = 0.0

        for (log_prob, value, _, entropy), R in zip(self.memory, returns):
            advantage = R - value.item()
            actor_loss  += -log_prob * advantage
            critic_loss += F.mse_loss(value.squeeze(), R.detach().clone())
            entropy_loss += -entropy

        loss = actor_loss + self.value_coef * critic_loss + self.entropy_coef * entropy_loss

        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
        self.optimizer.step()

        self.memory = []

    def reset_hidden(self):
        self.hidden = None

    def gen_features(self, obs):
        player = obs.get("player", 0)
        planets = obs.get("planets", [])
        fleets = obs.get("fleets", [])
        initial_planets = obs.get("initial_planets", [])
        angular_velocity = obs.get("angular_velocity", 0.0)
    
        features = [angular_velocity]
    
        enemy_planets = [p for p in planets if p[OWNER] != player and p[OWNER] != -1]
    
        for p in planets:
            dist_to_nearest_enemy = min(
                (math.hypot(p[X] - e[X], p[Y] - e[Y]) / 100.0 for e in enemy_planets),
                default=1.0
            )
            init = next((i for i in initial_planets if i[ID] == p[ID]), None)
            orbital_radius = math.hypot(init[X] - SUN_X, init[Y] - SUN_Y) / 100.0 if init else 1.0
    
            features.extend([
                float(p[OWNER] == player),
                float(p[OWNER] == -1),
                float(p[OWNER] != player and p[OWNER] != -1),
                p[X] / 100.0,
                p[Y] / 100.0,
                np.log1p(p[SHIPS]),
                p[PROD] / 5.0,
                math.hypot(p[X] - SUN_X, p[Y] - SUN_Y) / 100.0,
                orbital_radius,
                float(orbital_radius + p[RADIUS] >= (50/100)),
                dist_to_nearest_enemy,
            ])
    
        for f in fleets:
            features.extend([
                float(f[1] == player),
                f[2] / 100.0,
                f[3] / 100.0,
                np.log1p(f[6]),
                math.cos(f[4]),
                math.sin(f[4]),
            ])
    
        x = np.zeros(512)
        l = min(len(features), 512)
        x[:l] = features[:l]
    
        norm = np.linalg.norm(x)
        return x / (norm + 1e-8)

In [26]:
def train(n_games=100, pool_size=8, save_every=250):
    env = Environment()
    agent = A2CAgent(n_games=n_games)
    pool = []
    total_reward = 0.0
    wins = 0
    drawn = 0
    losts = 0

    pbar = tqdm(range(n_games))

    for game in pbar:
        agent.reset_hidden()

        if pool:
            opponent = random.choice(pool)
            opponent.reset_hidden()
        else:
            opponent = A2CAgent(n_games=n_games)
            opponent.model.load_state_dict(agent.model.state_dict())
            opponent.reset_hidden()

        episode_reward = 0.0
        for r1, r2, done in env(agent, opponent):
            agent.store_reward(r1)
            episode_reward += r1

        agent.update()
        agent.scheduler.step()
        total_reward += episode_reward

        result_0 = env.env.state[0].reward or 0
        if result_0 == 1:
            wins += 1
        elif result_0 == -1:
            losts += 1
        else:
            drawn += 1

        if game % save_every == 0 and game != 0:
            agent.save(f"orbit_agent_{game}.pt")
            snapshot = A2CAgent(n_games=n_games)
            snapshot.model.load_state_dict(agent.model.state_dict())
            pool.append(snapshot)
            if len(pool) > pool_size:
                pool.pop(0)

        pbar.set_postfix({
            "reward": f"{episode_reward:.2f}",
            "avg_reward": f"{total_reward / (game + 1):.2f}",
            "winrate": f"{wins / (game + 1):.2%}",
            "drawnrate": f"{drawn / (game + 1):.2%}",
            "loserate": f"{losts / (game + 1):.2%}",
            "pool": len(pool),
        })

In [ ]:
train(n_games=250, pool_size=6, save_every=50)

 48%|████▊     | 121/250 [11:32<16:07,  7.50s/it, reward=218.86, avg_reward=48.39, winrate=27.27%, drawnrate=0.00%, loserate=72.73%, pool=2]